# Aimilia feature ablation and Kaggle submission

**Owner: Leonard.** Reproduce the original 106-feature logistic regression ablation, then compare a bounded set of regularization, extra observation-only features, and nonlinear models on the same five development folds. Select by mean CV accuracy, evaluate the reserved 20% holdout once, and refit on all labels to generate `submission.csv`.

Run all cells using the project `.venv` kernel. The holdout is never used to select features or model settings. Small CV differences are descriptive, and this search can overfit development folds. These folds were already used in previous exploration, so the untouched holdout is the final local check. No Kaggle upload is performed. Artifacts include the executed notebook, exact settings, split hashes, metrics, final model, and submission-format verification.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/pokemon_challenge").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside FDS_Challenge_26.")
for folder in (ROOT / "src", ROOT / "notebooks/aimilia"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from pokemon_challenge.experiment import BattleExperiment, Candidate
from pokemon_challenge.models.classifiers import make_pipeline
from pokemon_challenge.data import iter_battles
from pokemon_challenge.paths import data_dir
import features_static as static
import features_timeline as timeline
import features_effects as effects
import features_roster as roster
if str(ROOT / "notebooks/leonard") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks/leonard"))
from ablation_extensions import battle_state_features
# Bound native thread pools for predictable local runtime.
from threadpoolctl import threadpool_limits
THREAD_LIMIT = threadpool_limits(limits=2)

MODEL = "logistic_regression"
MODEL_PARAMS = {"C": 1.0, "max_iter": 4000, "solver": "lbfgs"}
# Alternative: MODEL = "hist_gradient_boosting"; MODEL_PARAMS = {"early_stopping": False}
# Feature usefulness depends on the model. Changing it starts a separate comparison.
RUN = datetime.now().strftime("%Y%m%d_%H%M%S_%f_aimilia_feature_ablation")


## 1. Define interpretable groups

Keep her exact feature formulas. Split her timeline features into smaller groups so we can test health, status, activity, and moves separately. **Only boost features** from `features_effects.py` are included, matching her current notebook.

| Group | Inputs | Count |
|---|---|---:|
| static | Level-scaled starting stats, types, items, differences | 37 |
| health | Faints and active HP summaries/trends | 16 |
| status | Ailment and sleep rates | 5 |
| activity | Switching, species seen, no-move rates | 8 |
| moves | Mean power, accuracy, status-move rate | 6 |
| boosts | Mean, maximum, final and category-specific boost stages | 13 |
| roster | Estimated team HP and alive counts at turns 10, 20, 30 | 21 |

Roster health assumes unseen Pokémon are healthy and carries forward last-observed HP. These are estimates. Active HP changes also reflect switching, rather than only damage/healing.

In [ ]:
def observed_row(extractor, battle, **kwargs):
    # The shared runner removes IDs/labels before calling features.
    # Aimilia's functions require an ID for their dataframe builders; discard it here.
    row = extractor({**battle, "battle_id": 0}, **kwargs)
    return {k: v for k, v in row.items() if k not in ("battle_id", "player_won")}


def static_features(battle):
    return observed_row(static.battle_to_row, battle, labelled=False)


TIMELINE_KEYS = {
    "health": ("faints", "hppct_mean", "hppct_final", "hppct_min",
               "hppct_early_mean", "hppct_late_mean", "hppct_trend"),
    "status": ("status_rate", "slp_rate"),
    "activity": ("switches", "n_species_seen", "no_move_rate"),
    "moves": ("mean_base_power", "mean_accuracy", "status_move_rate"),
}
TIMELINE_DIFFS = {
    "health": ("hppct_mean", "hppct_final"),
    "status": ("status_rate",),
    "activity": ("no_move_rate", "switches"),
    "moves": (),
}


def timeline_group(battle, group):
    row = observed_row(timeline.battle_to_row, battle)
    keys = [f"{side}_{key}" for side in ("p1", "p2") for key in TIMELINE_KEYS[group]]
    keys += [f"diff_{key}" for key in TIMELINE_DIFFS[group]]
    return {key: row[key] for key in keys}


def health_features(battle):
    return timeline_group(battle, "health")


def status_features(battle):
    return timeline_group(battle, "status")


def activity_features(battle):
    return timeline_group(battle, "activity")


def move_features(battle):
    return timeline_group(battle, "moves")


def boost_features(battle):
    return {k: v for k, v in observed_row(effects.battle_to_row, battle).items() if "boost" in k}


def roster_features(battle):
    return observed_row(roster.battle_to_row, battle)


FEATURE_FUNCTIONS = {
    "static": static_features, "health": health_features,
    "status": status_features, "activity": activity_features,
    "moves": move_features, "boosts": boost_features, "roster": roster_features,
}
ALL_GROUPS = tuple(FEATURE_FUNCTIONS)

# Verify grouping preserves all 106 original values on an actual battle.
example = next(iter_battles(data_dir() / "train.jsonl"))
original = {
    **observed_row(static.battle_to_row, example, labelled=False),
    **observed_row(timeline.battle_to_row, example),
    **boost_features(example), **roster_features(example),
}
grouped = {}
for function in FEATURE_FUNCTIONS.values():
    values = function(example)
    assert not (set(grouped) & set(values)), "Feature assigned to multiple groups"
    grouped.update(values)
assert grouped == original and len(grouped) == 106
print("Verified: grouping preserves Aimilia's 106 feature values.")

def effect_features(battle):
    return {k: v for k, v in observed_row(effects.battle_to_row, battle).items() if "eff_" in k}

# Preserve ALL_GROUPS as the original ablation reference.
FEATURE_FUNCTIONS.update(effects=effect_features, battle_state=battle_state_features)
EXTENDED_GROUPS = tuple(FEATURE_FUNCTIONS)


In [ ]:
experiment = BattleExperiment(owner="leonard", run=RUN, feature_functions=FEATURE_FUNCTIONS)
assert sum(c.split("__")[0] in ALL_GROUPS for c in experiment.X) == 106
print("Model:", MODEL, MODEL_PARAMS)
print("Validation:", experiment.validation)
print("Split hash:", experiment.manifest["splits_sha256"])
display(pd.Series({g: sum(c.startswith(g + "__") for c in experiment.X)
                   for g in EXTENDED_GROUPS}, name="input_features").to_frame())
display(experiment.assignments.groupby(["partition", "validation_fold"]).size().rename("battles").to_frame())

# Retain the complete notebook and imported feature implementations with this run.
import shutil
shutil.copy2(ROOT / "notebooks/leonard/05_aimilia_feature_ablation.ipynb", experiment.output_path)
for module in (static, timeline, effects, roster):
    shutil.copy2(module.__file__, experiment.output_path)
shutil.copy2(ROOT / "notebooks/leonard/ablation_extensions.py", experiment.output_path)


## 2. Remove one group at a time

`all_106` is the reference. Each `without_...` candidate removes exactly one group. `original_72` removes both boosts and roster features to reconstruct her earlier feature set, using the same model and splits as every other candidate.

Accuracy is the selection metric. The shared runner also saves individual fold scores, fold standard deviation, feature counts, fit times, preprocessing selection frequencies, configurations and split assignments. Constants are removed inside folds, so the effective feature count can be smaller than the input count.

In [ ]:
def candidate(name, groups):
    return Candidate(name=name, groups=tuple(groups), model=MODEL,
                     percentile=100, params=dict(MODEL_PARAMS))


candidates = [candidate("all_106", ALL_GROUPS)]
candidates += [candidate(f"without_{group}", [g for g in ALL_GROUPS if g != group])
               for group in ALL_GROUPS]
candidates += [candidate("original_72", [g for g in ALL_GROUPS if g not in ("boosts", "roster")])]
results = experiment.compare(candidates)
display(results)

## 3. Read the paired changes

A **positive delta** means the reduced version scored better than all 106 features. One percentage point is `1.0` in the delta columns. Each fold is compared with the exact same reference fold.

Check how many folds improve and the spread of their differences. A tiny mean gain with mixed fold results is weak evidence. These folds overlap in training data, so their spread is descriptive and is **not** an independent-sample confidence interval or significance test. Fewer features with similar accuracy can still be a useful simplification. Feature groups can interact: two individually removable groups may be harmful to remove together.

In [ ]:
import matplotlib.pyplot as plt

FOLD_COLUMNS = [f"fold_{i + 1}" for i in range(experiment.validation["n_splits"])]


def paired_comparison():
    table = experiment.results.set_index("candidate")
    reference = table.loc["all_106", FOLD_COLUMNS].to_numpy(dtype=float)
    rows = []
    for name, row in table.iterrows():
        delta = row[FOLD_COLUMNS].to_numpy(dtype=float) - reference
        rows.append({
            "candidate": name, "input_features": int(row["input_features"]),
            "mean_selected_features": row["mean_selected_features"],
            "cv_accuracy": row["cv_accuracy"], "fold_std": row["fold_std"],
            "delta_pp": 100 * delta.mean(), "delta_fold_std_pp": 100 * delta.std(),
            "folds_better": int((delta > 1e-12).sum()),
            "folds_equal": int(np.isclose(delta, 0, atol=1e-12, rtol=0).sum()),
            **{f"delta_fold_{i + 1}_pp": 100 * d for i, d in enumerate(delta)},
        })
    return pd.DataFrame(rows).sort_values("delta_pp", ascending=False).reset_index(drop=True)


summary = paired_comparison()
summary.to_csv(experiment.output_path / "paired_ablation.csv", index=False)
display(summary.round(4))

plot = summary[summary.candidate != "all_106"].sort_values("delta_pp")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot.candidate, plot.delta_pp,
        color=["#27866b" if value >= 0 else "#b64a55" for value in plot.delta_pp])
ax.axvline(0, color="black", linewidth=1)
ax.set(xlabel="Accuracy change vs all 106 features (percentage points)",
       title="Positive: removing these features improved mean CV accuracy")
fig.tight_layout()
fig.savefig(experiment.output_path / "group_ablation.png", dpi=160)
plt.show()

## 4. Optional: test one combined removal

After reviewing the table, enter a small, justified combination below and enable it. Re-run the paired-comparison cell afterward to include the result. Do not assume gains from separate removals add together. Avoid trying every possible combination: repeated searching can overfit these validation folds too.

In [ ]:
RUN_COMBINED_REMOVAL = False
DROP_TOGETHER = ("moves", "boosts")  # Example only; choose from the evidence above.

if RUN_COMBINED_REMOVAL:
    if not DROP_TOGETHER or not set(DROP_TOGETHER) <= set(ALL_GROUPS):
        raise ValueError("Choose known groups to remove.")
    kept = tuple(g for g in ALL_GROUPS if g not in DROP_TOGETHER)
    if not kept:
        raise ValueError("Keep at least one group.")
    name = "without_combined_" + "_".join(sorted(set(DROP_TOGETHER)))
    if name in set(experiment.results.candidate):
        print("Already evaluated:", name)
    else:
        display(experiment.compare([candidate(name, kept)]))
    summary = paired_comparison()
    summary.to_csv(experiment.output_path / "paired_ablation.csv", index=False)
    display(summary.round(4))
else:
    print("Combined removal skipped.")

## 5. Optional: individual-feature permutation importance

This is slower. For each development fold, fit the all-feature reference on that fold's training rows. Shuffle one feature in its validation rows and measure the drop in accuracy. No holdout data is used. Five shuffles per feature and fold reduce shuffle noise.

A positive importance means shuffling hurt prediction. Near-zero or negative importance identifies candidates for investigation, **not an automatic deletion list**. Correlated features can substitute for one another; group ablation is often more informative. Permutation importance does not refit a model without that feature. Validate an actual removal separately before adopting it.

In [ ]:
RUN_PERMUTATION_IMPORTANCE = False
N_PERMUTATION_REPEATS = 5

if RUN_PERMUTATION_IMPORTANCE:
    from sklearn.inspection import permutation_importance

    X_dev = experiment.X.iloc[experiment.development]
    y_dev = experiment.y.iloc[experiment.development]
    importance_rows = []
    for fold, (train_index, validation_index) in enumerate(experiment.folds, start=1):
        print(f"Permutation importance: fold {fold}/{len(experiment.folds)}", flush=True)
        estimator = make_pipeline(MODEL, seed=experiment.validation["seed"],
                                  percentile=100, params=dict(MODEL_PARAMS))
        estimator.fit(X_dev.iloc[train_index], y_dev.iloc[train_index])
        importance = permutation_importance(
            estimator, X_dev.iloc[validation_index], y_dev.iloc[validation_index],
            scoring="accuracy", n_repeats=N_PERMUTATION_REPEATS,
            random_state=experiment.validation["seed"] + fold, n_jobs=1,
        )
        for feature, values in zip(X_dev.columns, importance.importances):
            for repeat, value in enumerate(values, start=1):
                importance_rows.append({"feature": feature, "fold": fold,
                                        "repeat": repeat, "accuracy_drop": value})
    raw_importance = pd.DataFrame(importance_rows)
    raw_importance.to_csv(experiment.output_path / "permutation_importance_raw.csv", index=False)
    # First average shuffles within each fold; then describe variation across folds.
    per_fold = raw_importance.groupby(["feature", "fold"]).accuracy_drop.mean().unstack("fold")
    ranked = pd.DataFrame({
        "mean_accuracy_drop_pp": 100 * per_fold.mean(axis=1),
        "fold_std_pp": 100 * per_fold.std(axis=1, ddof=0),
        "folds_positive": (per_fold > 0).sum(axis=1),
    }).sort_values("mean_accuracy_drop_pp", ascending=False)
    ranked.to_csv(experiment.output_path / "permutation_importance.csv")
    print("Most important features:")
    display(ranked.head(20))
    print("Lowest importance — investigate redundancy/noise before removing:")
    display(ranked.tail(20).sort_values("mean_accuracy_drop_pp"))
else:
    print("Permutation importance skipped; enable it after the group comparison if useful.")

## 6. Bounded improvement experiment

Keep the original features: prior ablation did not support deleting a group. Test stronger/weaker logistic regularization, the omitted effects, and last-seen roster status plus recent faint/HP changes. Same-species consecutive HP changes avoid counting switches as damage/healing. Last-seen opponent information remains an estimate. No outcome, ID, future turn, or dataset-fitted statistic is used as a feature.

Compare nonlinear histogram boosting (two tree complexities on extended features, one original-feature control) and a regularized random forest. Every candidate uses the same development folds and fold-fitted preprocessing; choose the highest mean accuracy with deterministic ordering to break exact ties. This is a bounded search, not evidence that the largest feature set or any model family is always better.


In [ ]:
improvement_candidates = [
    Candidate("lr_C01", ALL_GROUPS, params={**MODEL_PARAMS, "C": 0.1}),
    Candidate("lr_C10", ALL_GROUPS, params={**MODEL_PARAMS, "C": 10.0}),
    Candidate("lr_with_effects", (*ALL_GROUPS, "effects"), params=MODEL_PARAMS),
    Candidate("lr_extended", EXTENDED_GROUPS, params={**MODEL_PARAMS, "C": 0.1}),
    Candidate("hgb_original", ALL_GROUPS, model="hist_gradient_boosting",
              params={"max_iter": 200, "learning_rate": 0.06, "max_leaf_nodes": 15,
                      "min_samples_leaf": 40, "l2_regularization": 5.0, "early_stopping": False}),
    Candidate("hgb_extended", EXTENDED_GROUPS, model="hist_gradient_boosting",
              params={"max_iter": 200, "learning_rate": 0.06, "max_leaf_nodes": 15,
                      "min_samples_leaf": 40, "l2_regularization": 5.0, "early_stopping": False}),
    Candidate("hgb_extended_small", EXTENDED_GROUPS, model="hist_gradient_boosting",
              params={"max_iter": 160, "learning_rate": 0.05, "max_leaf_nodes": 7,
                      "min_samples_leaf": 60, "l2_regularization": 10.0, "early_stopping": False}),
    Candidate("rf_extended", EXTENDED_GROUPS, model="random_forest",
              params={"n_estimators": 350, "max_features": 0.7, "min_samples_leaf": 8,
                      "max_depth": 16, "n_jobs": 2}),
]
display(experiment.compare(improvement_candidates))
comparison = paired_comparison()
comparison.to_csv(experiment.output_path / "all_candidate_comparison.csv", index=False)
display(comparison.round(4))


## 7. Freeze the CV winner and evaluate the holdout once

Use the CV table to choose the final candidate. Prefer consistent gains or a simpler version with comparable accuracy. The default suggestion below picks the highest mean CV score; override it when a tiny difference does not justify extra complexity.

Holdout evaluation below runs **after all feature/model choices are finished**. The shared runner then locks further comparisons. Do not create a new run to continue tuning against the same holdout. The holdout score estimates performance of the selected procedure; it is not another feature-selection metric.

The shared split currently stratifies ordinary battle rows. Related or duplicate battles can cross partitions; investigate their origin before making a strong generalisation claim. Dropping duplicates merely because of their effect on CV is not a sound selection rule.

In [ ]:
SUGGESTED_CANDIDATE = experiment.best_candidate
print("Highest mean CV score:", SUGGESTED_CANDIDATE)
CHOSEN_CANDIDATE = SUGGESTED_CANDIDATE  # Override with a candidate name if justified.
EVALUATE_HOLDOUT = True

if EVALUATE_HOLDOUT:
    holdout = experiment.evaluate_holdout(CHOSEN_CANDIDATE)
    display(pd.Series({k: holdout[k] for k in ("candidate", "accuracy", "n_battles")}))
else:
    print("Holdout remains untouched. Finish CV decisions before enabling evaluation.")

print("Saved results:", experiment.output_path)
print("Main files: paired_ablation.csv, cv_results.csv, group_ablation.png, splits.csv")


## 8. Full-data fit and validated Kaggle CSV

After the single holdout evaluation, refit the frozen CV-selected configuration on every labelled battle. Predictions are binary labels aligned exactly to the official sample IDs. The CSV contains no index column. The local holdout score is an estimate, not a promised Kaggle score.


In [ ]:
submission = experiment.make_submission()
sample = pd.read_csv(experiment.data_path / "sample_submission.csv")
assert list(submission.columns) == ["battle_id", "player_won"]
assert submission.battle_id.equals(sample.battle_id)
assert submission.battle_id.is_unique and submission.notna().all().all()
assert submission.player_won.isin([0, 1]).all()
reloaded = pd.read_csv(experiment.output_path / "submission.csv")
pd.testing.assert_frame_equal(submission, reloaded)
report = {
    "selected_candidate": CHOSEN_CANDIDATE,
    "baseline_cv_accuracy": float(experiment.results.set_index("candidate").loc["all_106", "cv_accuracy"]),
    "selected_cv_accuracy": float(experiment.results.iloc[0].cv_accuracy),
    "holdout_accuracy": holdout["accuracy"],
    "submission_rows": len(submission),
    "prediction_counts": submission.player_won.value_counts().to_dict(),
    "submission": str(experiment.output_path / "submission.csv"),
    "format_checks_passed": True,
}
(experiment.output_path / "run_summary.json").write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))
display(submission.head())


## Completed run: 20261101_230041_643033

Selected **hgb_extended** from 17 candidates: **66.475%** mean CV accuracy vs **66.300%** for the original all-106 logistic baseline (+0.175 percentage points). Single reserved holdout: **67.75%** (1,355/2,000). This is a small CV gain, not proof of a reliable improvement. Final model fitted on all 10,000 labels; `submission.csv` contains 5,000 validated predictions in official sample order.

The exact-observation audit found 14 repeated observations with consistent labels; four duplicate groups cross development/holdout and seven cross development folds. Splits were retained to compare with prior ablation, so local scores may be slightly optimistic. Agree on duplicate-grouped splits and a new unused evaluation set before further tuning. Exact settings, scores and audit are in the run folder; the experiment record is `experiments/leonard/20261101_aimilia_ablation_submission.md`.
